In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import os
import cv2
from PIL import Image

In [2]:
IMG_SIZE = (224, 224)  # MobileNet standard default size
BATCH_SIZE = 32
NUM_CLASSES = 5

In [9]:
def get_dataset(directory,type_of_data="train",IMAGE_SIZE=(48,48),num_classes=43):
    X = [] 
    y = [] 
    
    # Iterate through image folders
    for i in range(num_classes):
        path = os.path.join(directory,type_of_data,str(i)) 
        
        # List all the images in the folder
        for j in os.listdir(path):  
            
            # Open the image
            image = Image.open(path + '/'+ j) 
            
            # Resize it to (48,48)
            image = image.resize(IMAGE_SIZE) 
            
            # Convert it to a numpy array for easier use
            image = np.array(image) 
            
            # Add the image and the labels to the lists
            X.append(image) 
            y.append(i)
            
    return np.array(X), np.array(y)

In [10]:
def load_image(path,rgba=False):
    loaded_image = Image.open(path)
    if rgba:
        loaded_image = loaded_image.convert("RGB")
    loaded_image = loaded_image.resize((48,48))
    loaded_image = np.array(loaded_image)
    loaded_image = np.expand_dims(loaded_image,axis=0)
    return loaded_image

In [11]:
def calculate_results(y_true, y_pred):
    # Calculate model accuracy
    model_accuracy = accuracy_score(y_true, y_pred) * 100
    # Calculate model precision, recall and f1 score using "weighted average
    model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
    model_results = {"accuracy": model_accuracy,
                  "precision": model_precision,
                  "recall": model_recall,
                  "f1": model_f1}
    return model_results

In [8]:
def recognize_feature(image_path, model):
    # Read, resize, and match MobileNet dimensions
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    
    # Expand dimensions for a batch format: (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Scale raw pixel intensities down into the exact -1 to 1 model criteria
    processed_img = preprocess_input(img_batch)
    
    # Generate distribution probabilities
    predictions = model.predict(processed_img)
    class_index = np.argmax(predictions[0])
    confidence = predictions[0][class_index]
    
    return class_index, confidence

In [17]:
directory = "/Users/maliya/Desktop/dissertation/Vista/data/raw/consolidated/gtsrb-german-traffic-sign"
IMAGE_SIZE = (96,96)
SMALL_IMAGE_SIZE = (48,48) 
num_classes = 43
sign_names = ["limit_zone_20","limit_zone_30","limit_zone_50","limit_zone_60","limit_zone_70",
              "limit_zone_80","end_of_speed_limit","limit_zone_100","limit_zone_120",
              "no_passing","no_passing_for_trucks","right_of_way","priority_road",
              "yield_right_of_way","stop","prohibited_for_all_vehicles","tractors_and_trucks_prohibited",
              "entry_prohibited","danger","single_curve_left","single_curve_right","double_curve",
              "rough_road","slippery_road","road_narrows","construction_site","signal_lights_ahead","pedestrian_crosswalk_ahead",
              "children","bicycle_crossing","snow_ahead","wild_animal_crossing","end_of_all_restrictions",
               "mandatory_right","mandatory_left","mandatory_ahead","mandatory_ahead_right",
              "mandatory_ahead_left","mandatory_down_right","mandatory_down_left","traffic_circle","end_of_no_passing_zone",
              "end_of_no_passing_zone_trucks"]
train_data, train_labels = get_dataset(directory=directory,
                                      type_of_data="train",
                                      IMAGE_SIZE=SMALL_IMAGE_SIZE,
                                      num_classes=num_classes)

In [22]:
x_train, x_valid, y_train, y_valid = train_test_split(train_data, 
                                                    train_labels, 
                                                    test_size=0.2, 
                                                    random_state=42)
y_train = to_categorical(y_train, 43)
y_valid = to_categorical(y_valid, 43)
train_dataset = tf.data.Dataset.zip((
    tf.data.Dataset.from_tensor_slices(x_train),
    tf.data.Dataset.from_tensor_slices(y_train)
))

valid_dataset = tf.data.Dataset.zip((
    tf.data.Dataset.from_tensor_slices(x_valid),
    tf.data.Dataset.from_tensor_slices(y_valid)
))

In [23]:
def normalize_img(image,label):
    return tf.cast(image,tf.float32)/255., label

#Apply normalization and prefetching
train_dataset = train_dataset.map(normalize_img,num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)
valid_dataset = valid_dataset.map(normalize_img,num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

In [49]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3), 
    include_top=False, 
    weights='imagenet'
)
base_model.trainable = False
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),  # Compress feature map spatial dimensions
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),              # Mitigate overfitting risks
    layers.Dense(NUM_CLASSES, activation='softmax') # Outputs target probabilities
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# Resize images to match MobileNetV2 input size (224, 224)
def resize_to_mobilenet(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

train_dataset_resized = train_dataset.map(resize_to_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
valid_dataset_resized = valid_dataset.map(resize_to_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)

# Update NUM_CLASSES to match the actual number of traffic sign classes
NUM_CLASSES = 43

# Recompile the model with the correct number of output classes
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_18     │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,499 (9.26 MB)

 Trainable params: 169,515 (662.17 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [48]:
from pathlib import Path
from PIL import Image

directory = "/Users/maliya/Desktop/dissertation/Vista/data/raw/consolidated/gtsrb-german-traffic-sign"
dataset_dir = Path(directory)

# Loop through all images matching standard formats
extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
image_paths = []

for ext in extensions:
    image_paths.extend(dataset_dir.glob(ext))
loaded_images = []
for path in image_paths:
    class_index, confidence = recognize_feature(path, model)
    print(class_index, confidence)
    img = Image.open(path)
    loaded_images.append(img)
    dominant_color(img)
    print(bgr_to_name(img))


In [ ]:
reduce_lr_callback = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
)
early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
checkpoint_callback = ModelCheckpoint(
    "best_model.h5", monitor="val_loss", save_best_only=True
)

In [ ]:
history = model.fit(
    train_dataset_resized,
    validation_data=valid_dataset_resized,
    epochs=10,
    callbacks=[reduce_lr_callback, early_stopping_callback, checkpoint_callback]
)

In [ ]:
import coremltools as ct

# coremltools' TF2 frontend can't convert a Keras 3 model object or a raw
# ConcreteFunction with captured resource variables directly, so export to a
# SavedModel with a single serving signature and convert from that instead.
saved_model_dir = "mobilenetv2_saved_model"
model.export(saved_model_dir)

loaded = tf.saved_model.load(saved_model_dir)
serving_fn = loaded.signatures["serving_default"]

single_sig_dir = "mobilenetv2_saved_model_single"
tf.saved_model.save(loaded, single_sig_dir, signatures={"serving_default": serving_fn})

input_name = list(serving_fn.structured_input_signature[1].keys())[0]

converted_model = ct.convert(
    single_sig_dir,
    source="tensorflow",
    inputs=[
        ct.TensorType(
            name=input_name,
            shape=(1, IMG_SIZE[0], IMG_SIZE[1], 3),
            dtype=np.float32,
        )
    ],
)

converted_model.save("MobileNetV2.mlpackage")

In [40]:
def dominant_color(image_bgr, k=3):
    """Return the most common color in an image region as (B, G, R)."""
    # Downsample for speed — exact pixel count doesn't matter for a color average.
    small = cv2.resize(image_bgr, (50, 50), interpolation=cv2.INTER_AREA)
    pixels = small.reshape(-1, 3).astype(np.float32)
    kmeans = KMeans(n_clusters=min(k, len(pixels)), n_init=4, random_state=0)
    labels = kmeans.fit_predict(pixels)
    # The cluster with the most assigned pixels is the dominant color.
    counts = np.bincount(labels)
    dominant = kmeans.cluster_centers_[np.argmax(counts)]
    return tuple(int(c) for c in dominant) 

In [41]:
def bgr_to_name(bgr):
    """Very rough color-name bucketing — swap for a proper color-name lookup
    (e.g. webcolors, or a nearest-neighbor match against a named-color table)
    if you need accurate names rather than raw RGB."""
    b, g, r = bgr
    if max(r, g, b) < 60:
        return "black"
    if min(r, g, b) > 200:
        return "white"
    if r > g and r > b:
        return "red" if r - max(g, b) > 40 else "pink/brown"
    if g > r and g > b:
        return "green"
    if b > r and b > g:
        return "blue"
    if r > 150 and g > 150 and b < 100:
        return "yellow"
    return "gray/mixed"